In [1]:
import scipp as sc
import scippnexus as snx
import plopp as pp
import numpy as np

from easydynamics.Job import Job
from easydynamics.Experiment import Experiment
from easydynamics.Experiment import Data

from easydynamics.sample import BrownianTranslationalDiffusion
from easydynamics.sample import JumpDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import Lorentzian
from easydynamics.sample import DeltaFunction
from easydynamics.sample import Polynomial
# from easydynamics.sample import DampedHarmonicOscillator


from easydynamics.sample import Gaussian



%matplotlib widget
# data_path_lowT = r"C:\Users\henrikjacobsen3\Dropbox\DMSC\Halric 2025\new_IRIS_data\iris_Cells_Plus_TCZ_108330_to108389_Plus_Empty_Data_SQW"
data_path = r"C:\Users\henrikjacobsen3\Dropbox\DMSC\Halric 2025\IRIS_data_vanadium_norm"


In [2]:
# filenumber = 108556 # Empty container long

# filename = data_path + fr"/iris{filenumber}_graphite002_sqw.nxs"
filename = data_path + "/iris108556-108557_multi_graphite002_sqw.nxs"
all_data = snx.load(filename)

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

In [3]:
all_data

DataGroup(sizes={'dim_0': None, 'dim_1': None, 'detector_number': 110, 'time': None, 'axis2': 27, 'axis1': 2000}, keys=[
    mantid_workspace_1: DataGroup(9, {'dim_0': None, 'dim_1': None, 'detector_number': 110, 'time': None, 'axis2': 27, 'axis1': 2000}),
])

In [4]:
# Load data function
number_of_Q_bins = 10 # we may change this later, but this seems like a reasonable start
number_of_energy_bins = 501

# def load_data(filenumber: int, number_of_Q_bins, number_of_energy_bins):
def load_data(filename, number_of_Q_bins, number_of_energy_bins, data_path=data_path):
    if isinstance(filename, int):
        filename = data_path + fr"/iris{filename}_graphite002_sqw.nxs"
    else:
        filename = data_path + "/" + filename
    all_data = snx.load(filename)

    temperature = all_data['mantid_workspace_1']['logs']['Sample']
    temperature.coords['time'].unit = 's'
    temperature.unit = 'K'

    data=all_data['mantid_workspace_1']['workspace'].rename({'axis1': 'energy', 'axis2': 'Q'})

    data.coords['energy'].unit = 'meV'
    data.coords['Q'].unit = '1/Angstrom'

    del data.coords['frac_area']

    # Rebin data and use midpoints instead of edges
    rebinned_data=data.rebin(Q=number_of_Q_bins, energy=number_of_energy_bins)

    rebinned_data.coords['Q'] = sc.midpoints(rebinned_data.coords['Q'])
    rebinned_data.coords['energy'] = sc.midpoints(rebinned_data.coords['energy'])


    # Remove 0 variances
    v = rebinned_data.variances
    v[v <= 0] = 1.0


    return rebinned_data, temperature




In [6]:
filename="iris108556-108557_multi_graphite002_sqw.nxs"
EC_data,EC_temperature=load_data(filename, number_of_Q_bins, number_of_energy_bins, data_path=data_path)

filename = "iris108388_graphite002_sqw.nxs"
data,temperature_data=load_data(filename, number_of_Q_bins, number_of_energy_bins, data_path=data_path)



c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

In [7]:
#  data_and_model = {"Data": self._experiment._data.data, "Model": fit_total}
#         if plot_individual_components and component_arrays:
#             data_and_model.update(component_arrays)
#         data_and_model = sc.DataGroup(data_and_model)

rescaled_EC_data = EC_data.copy()*0.4

substracted_data = data - rescaled_EC_data

data_and_EC = {"Data": data, "Empty Container": EC_data, "Rescaled Empty Container": rescaled_EC_data, "Data - Rescaled EC": substracted_data}
data_and_EC = sc.DataGroup(data_and_EC)

pp.slicer(data_and_EC["energy", -0.2*sc.Unit("meV"):0.2*sc.Unit("meV")])


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [ ]:
# Now let's use the low temperature data to estimate the resolution.
resolution_job= Job(name='resolution')


filenumber = 108331 # base T long
lowt_data,lowt_temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins,data_path_lowT)

exp=Experiment()
res_data=Data()
res_data.append(lowt_data)

exp.set_data(res_data)

resolution_job.set_experiment(exp)


bg=SampleModel('Background')
bg.add_component(Polynomial(coefficients=[1e-3]))
resolution_job.set_background_model(bg)


resolution_model=SampleModel(name="ResolutionModel")
resolution_model.add_component(Gaussian(name="Res1", area=1.5,width=0.01))
resolution_model.add_component(Gaussian(name="Res2", area=1.0,width=0.015,center=-0.01))
resolution_model.add_component(Lorentzian(name="Res3", area=0.3,width=0.015,center=-0.025))

resolution_job.set_theory(resolution_model)
resolution_job.generate_analysis_for_cuts()
for i in range(len(resolution_job.analysis)):
    resolution_job.analysis[i].get_fit_parameters()[8].min=0 #polynomial
    resolution_job.analysis[i].get_fit_parameters()[0].min=0 #Gauss 1 area
    resolution_job.analysis[i].get_fit_parameters()[2].min=0 #Gauss 2 area

resolution_job.fit(sequential = "Q")


In [ ]:

resolution_job.plot_data_and_model(intensity_min=0.0, intensity_max=120.0,
                            energy_min=-0.2, energy_max=0.2)

In [ ]:
resolution_job._analysis[1].get_fit_parameters()

a=resolution_job.return_data_and_model()
a

In [ ]:

def fit_simultaneous(filenumber,number_of_Q_bins,number_of_energy_bins):
    data, temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins)
    Cells_Plus_TCZ_simultaneous= Job(name='JumpDiffusion')


    exp=Experiment()
    highT_data_simul=Data()
    highT_data_simul.append(data)

    exp.set_data(highT_data_simul)

    Cells_Plus_TCZ_simultaneous.set_experiment(exp)
    Cells_Plus_TCZ_simultaneous.generate_empty_analysis_array()


    bg=SampleModel('Background')
    bg.add_component(Polynomial(coefficients=[1e-3]))
    Cells_Plus_TCZ_simultaneous.set_background_model(bg)
    Cells_Plus_TCZ_simultaneous.set_background_model_for_all_analyses()
    T_motion=249.1
    avg_T = sc.mean(temperature)
    if avg_T.value>T_motion:
        diffusion_model=JumpDiffusion(name="JumpDiffusion", diffusion_coefficient=1.8, tau = 5.0,scale=0.2)
        Cells_Plus_TCZ_simultaneous.set_diffusion_model(diffusion_model)
    # delta_model=DeltaFunction(name="Delta",area=0.05)
    # Cells_Plus_TCZ_simultaneous.set_theory_for_all_analyses(delta_model) # Something wrong here, it uses the same parameter
    # Cells_Plus_TCZ.use_fit_as_resolution(resolution_job) # doesn't seem to work for some reason
    for i in range(len(Cells_Plus_TCZ_simultaneous._analysis)):
        this_resolution_model=resolution_job._analysis[i]._theory
        Cells_Plus_TCZ_simultaneous._analysis[i].set_resolution_model(this_resolution_model)
        Cells_Plus_TCZ_simultaneous._analysis[i].fix_resolution_parameters()
        if avg_T.value>T_motion:
            Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(DeltaFunction(name="Delta",area=0.05))
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[5].min=1e-8 # BG
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[2].min=0.001   #D     
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[0].min=0.0  #scale      
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=0.001  #tau 
        else:
            theory=SampleModel()
            theory.add_component(DeltaFunction(name="Delta",area=1.85))
            Cells_Plus_TCZ_simultaneous._analysis[i].set_theory(theory)
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[1].min=1e-8 # BG

    result=Cells_Plus_TCZ_simultaneous.fit_simultaneous()

    return Cells_Plus_TCZ_simultaneous,result,temperature


def fit_sequential(filenumber,number_of_Q_bins,number_of_energy_bins,data_path=data_path):
    data, temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins,data_path)
    Cells_Plus_TCZ_simultaneous= Job(name='JumpDiffusion')


    exp=Experiment()
    highT_data_simul=Data()
    highT_data_simul.append(data)

    exp.set_data(highT_data_simul)

    Cells_Plus_TCZ_simultaneous.set_experiment(exp)
    Cells_Plus_TCZ_simultaneous.generate_empty_analysis_array()


    bg=SampleModel('Background')
    bg.add_component(Polynomial(coefficients=[1e-3]))
    Cells_Plus_TCZ_simultaneous.set_background_model(bg)
    Cells_Plus_TCZ_simultaneous.set_background_model_for_all_analyses()
    T_motion=249.1
    avg_T = sc.mean(temperature)
    # if avg_T.value>T_motion:
        # diffusion_model=JumpDiffusion(name="JumpDiffusion", diffusion_coefficient=1.8, tau = 5.0,scale=0.2)
        # Cells_Plus_TCZ_simultaneous.set_diffusion_model(diffusion_model)
    # delta_model=DeltaFunction(name="Delta",area=0.05)
    # Cells_Plus_TCZ_simultaneous.set_theory_for_all_analyses(delta_model) # Something wrong here, it uses the same parameter
    # Cells_Plus_TCZ.use_fit_as_resolution(resolution_job) # doesn't seem to work for some reason
    for i in range(len(Cells_Plus_TCZ_simultaneous._analysis)):
        this_resolution_model=resolution_job._analysis[i]._theory
        Cells_Plus_TCZ_simultaneous._analysis[i].set_resolution_model(this_resolution_model)
        Cells_Plus_TCZ_simultaneous._analysis[i].fix_resolution_parameters()
        if avg_T.value>T_motion:
            theory=SampleModel()
            theory.add_component(DeltaFunction(name="Delta",area=0.05))
            theory.add_component(Lorentzian(name="Lorentzian",area=0.3, width=0.1))
            Cells_Plus_TCZ_simultaneous._analysis[i].set_theory(theory)
            # Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(DeltaFunction(name="Delta",area=0.05))
            # Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(Lorentzian(name="Lorentzian",area=0.3, width=0.1))
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=1e-10 # BG
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[2].min=0.001   #D     
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[0].min=0.0  #scale      
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=0.001  #tau 
        else:
            theory=SampleModel()
            theory.add_component(DeltaFunction(name="Delta",area=1.85))
            theory.add_component(Lorentzian(name="Lorentzian",area=0.0, width=1.0))
            Cells_Plus_TCZ_simultaneous._analysis[i].set_theory(theory)
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[1].min=1e-10 # BG
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=1e-10 # BG
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[2].fixed=True # don't fit the Lorentzian at low temperature, but keep it for plotting purposes
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[1].fixed=True # don't fit the Lorentzian at low temperature, but keep it for plotting purposes

    result=Cells_Plus_TCZ_simultaneous.fit()

    return Cells_Plus_TCZ_simultaneous,result,temperature


In [ ]:
this_job,this_result,this_temperature = fit_sequential(filenumber,number_of_Q_bins,number_of_energy_bins,data_path_lowT)


In [ ]:
this_job._analysis[0].get_fit_parameters()

In [ ]:
# filenumbers = range(108330,108390)
filenumbers = ["iris108388_graphite002_sqw.nxs"]
job_list=[]
temperature_list=[]
number_of_Q_bins = 10 # we may change this later, but this seems like a reasonable start
number_of_energy_bins = 501
for filenumber in filenumbers:
    this_job,this_result,this_temperature = fit_sequential(filenumber,number_of_Q_bins,number_of_energy_bins)
    job_list.append(this_job)
    temperature_list.append(this_temperature)
    print(this_temperature)
    print(this_job.analysis[5].get_fit_parameters())


this_job.plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

In [ ]:
this_job.plot_fit_parameters(parameter_name ="Delta area" )


In [ ]:
this_job.plot_fit_parameters(parameter_name ="Lorentzian area" )


In [ ]:

this_job.plot_fit_parameters(parameter_name ="Lorentzian width" )

In [ ]:
job_list[54].plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

In [ ]:
job_list[49]._analysis[5].get_fit_parameters()
# job_list[49].get_parameters_as_data_group()

In [ ]:
    
# temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in temps]), unit = 'K')
# # Combine all data arrays into a single DataArray along a new dimension "temperature"
# all_data = sc.concat(datas, dim='temperature')
# all_data.coords['temperature'] = temperature


In [ ]:
# P1=job_list[49].get_parameters_as_data_group()
# P2=job_list[50].get_parameters_as_data_group()
# P=sc.concat([P1,P2],dim='temperature')
groups = [job.get_parameters_as_data_group() for job in job_list]
fit_parameters = sc.concat(groups, dim='temperature')
avg_temperature=[sc.mean(t) for t in temperature_list]
temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in avg_temperature]), unit = 'K')

# fit_parameters.coords['temperature']=temperature

In [ ]:
a=fit_parameters['Delta area']['value']
a
a.coords['temperature']=temperature
pp.slicer(a,vmin=0,vmax=3)

In [ ]:
width=fit_parameters['Lorentzian width']['value']
width.coords['temperature']=temperature

area=fit_parameters['Lorentzian area']['value']
area.coords['temperature']=temperature


In [ ]:
pp.slicer(width,vmin=0,vmax=0.2)

In [ ]:
pp.slicer(area)

In [ ]:
job_list[50].plot_data_and_model(intensity_min=-0.5, intensity_max=80,
                            energy_min=-0.3, energy_max=0.5)

In [ ]:
# resolution_job.return_data_and_model()

groups = [job.return_data_and_model() for job in job_list]
fit_parameters = sc.concat(groups, dim='temperature')
# avg_temperature=[sc.mean(t) for t in temperature_list]
# temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in avg_temperature]), unit = 'K')

linestyle = {"Data": "none", "Model": "-", "Lorentzian": "--", "Polynomial":"--", "Delta":"--"}
marker = {"Data": "o", "Model": "none", "Lorentzian": "none", "Polynomial":"none", "Delta":"none"}
markerfacecolor = {"Data": "none", "Model": "none", "Lorentzian": "none", "Polynomial":"none", "Delta":"none"}
color = {"Data": "black", "Model": "red", "Lorentzian": "green", "Polynomial":"purple", "Delta": "blue"}


pp.slicer(fit_parameters,keep='energy',vmin=-0.5,vmax=20,linestyle=linestyle, marker=marker,markerfacecolor=markerfacecolor, color=color)
# fit_parameters